# Lab 03 - Reliability Boosters for Prompts

**Week 3, Prompt Engineering and Task-to-Prompt Mapping**

You will make prompts reliable and auditable by adding clear delimiters,
explicit refusal guidance, an Answer to Verify workflow, rubric-based scoring,
and controlled determinism levers.

### Outcomes
By the end you can:
1. Use delimiters to isolate instruction, context, inputs, and contract, and prevent mixing.
2. Enforce a JSON output contract with a validator that reports every violation.
3. Implement a two-pass Answer to Verify workflow that repairs a flawed draft.
4. Build a rubric-based scorer that grades a triage output 0 to 10.
5. Explain and demonstrate temperature and top_p trade-offs for determinism versus creativity.

### How this lab runs
Everything runs offline against a small deterministic mock model, so results
are reproducible and no API key is needed. The final cell wires the same
pattern to the real Anthropic API for reference.

### How the checks work
Each task has a soft `check(...)` cell that prints PASS or FAIL and never
crashes. You start red. Fill each `raise NotImplementedError` with a working
body until every check reads PASS. Run `summary()` at the end for your score.

### Responsible AI, woven in
The refusal guidance is the safety spine of this lab. When evidence is thin the
correct move is to label `unknown` and ask for one specific detail, never to
guess. The rubric rewards that discipline, and the verifier enforces it.


In [ ]:
# --- Lab 03 setup: fixed data, soft check helper, provided mock model ---
import json
import re
import random

# K-Doc: the support knowledge excerpt. Line numbers are 1-based.
K_DOC = [
    "Password reset emails may be delayed up to 15 minutes.",
    "A token invalid message often means the link was used already or expired after 24h.",
    "Billing disputes should be escalated if a duplicate charge is over 48h unresolved.",
    "For Android 14 export crashes, the known fix ships in the v5.2.1 patch.",
]

# The three support inputs, keyed by id.
INPUTS = {
    "I-1": "Reset link says token invalid, how do I fix this?",
    "I-2": "My card was charged twice last week. No refund yet.",
    "I-3": "App crashes when exporting to PDF on my Pixel 8 (Android 14).",
}

ALLOWED = {"account", "billing", "bug", "unknown"}
WORD = re.compile(r"\b\w+\b")           # word tokenizer used for length checks

# Policy tables (product decisions, provided for you).
FALLBACK_ACTIONS = {
    "account": "Ask the user to request a new reset link and confirm the exact error text",
    "billing": "Escalate the duplicate charge for a refund review since it remains unresolved past 48 hours",
    "bug": "Advise the user to install the latest patch that resolves the Android export crash issue",
    "unknown": "Please share the exact error text and the last four account digits to proceed",
}

LABEL_SYNONYMS = {
    "account": "account", "account_issue": "account", "login": "account",
    "billing": "billing", "payment": "billing", "refund": "billing",
    "bug": "bug", "crash": "bug", "defect": "bug",
    "unknown": "unknown",
}

# ---- soft check helper: never raises, so a red notebook has zero crashes ----
_RESULTS = []

def check(name, test):
    """Run test() and record PASS or FAIL. test is a zero-arg callable that
    returns a truthy value on success. Any exception counts as FAIL."""
    try:
        ok = bool(test())
        note = ""
    except NotImplementedError:
        ok, note = False, "not implemented"
    except Exception as e:
        ok, note = False, type(e).__name__ + ": " + str(e)
    _RESULTS.append((name, ok))
    tag = "PASS" if ok else "FAIL"
    print(f"[{tag}] {name}" + (f"   ({note})" if note and not ok else ""))
    return ok

def summary():
    passed = sum(1 for _, ok in _RESULTS if ok)
    total = len(_RESULTS)
    print(f"\n=== {passed} of {total} checks passing ===")


In [ ]:
# --- Provided: a deterministic mock triage model (stands in for a hosted LLM) ---
# You do not edit this cell. It lets the lab run offline and reproducibly.
class MockTriageModel:
    """Deterministic stand-in for a hosted triage LLM. Classifies each input
    against the K-Doc with fixed keyword rules. A temperature knob controls
    phrasing variation so the determinism lesson is observable offline."""

    CANON = {
        "account": ("Ask the user to request a fresh reset link and retry within 24 hours", 2),
        "billing": ("Escalate the duplicate charge for refund since it stays unresolved beyond 48 hours", 3),
        "bug": ("Advise the user to update to the patch that fixes Android export crashes", 4),
        "unknown": ("Please share the exact error text and last four digits to continue", "none"),
    }
    PHRASE_POOL = {
        "account": [
            "Ask the user to request a fresh reset link and retry within 24 hours",
            "Have the user generate a new reset link then try again shortly after",
            "Request a brand new reset link from the user and retry within a day",
        ],
        "billing": [
            "Escalate the duplicate charge for refund since it stays unresolved beyond 48 hours",
            "Send the duplicate charge to escalation because it has stayed open past 48 hours",
            "Route this duplicate charge to a refund review since it remains open two days",
        ],
        "bug": [
            "Advise the user to update to the patch that fixes Android export crashes",
            "Tell the user to install the patch release that resolves Android export crashes",
            "Point the user to the patch that repairs the Android export crash behavior",
        ],
    }

    def classify(self, text):
        low = text.lower()
        if any(w in low for w in ("charge", "charged", "refund", "billing", "card")):
            return "billing", 3
        if any(w in low for w in ("crash", "export", "android", "pixel")):
            return "bug", 4
        if any(w in low for w in ("reset", "token", "password", "login", "link")):
            return "account", 2
        return "unknown", "none"

    def _extract_inputs(self, prompt):
        block = re.search(r"<INPUTS>(.*?)</INPUTS>", prompt, re.DOTALL)
        body = block.group(1) if block else prompt
        return {m.group(1): m.group(2).strip()
                for m in re.finditer(r"(I-\d+):\s*(.+)", body)}

    def respond(self, prompt):
        """Full-prompt path (Part B). Emits contract JSON when the prompt
        contains a CONTRACT block, else a freeform bullet list."""
        found = self._extract_inputs(prompt)
        records = []
        for iid, text in found.items():
            label, ev = self.classify(text)
            action, canon_ev = self.CANON[label]
            evidence = str(ev if ev != "none" else canon_ev)
            records.append({"input_id": iid, "label": label,
                            "next_action": action, "evidence": evidence})
        if "<CONTRACT>" in prompt:
            return json.dumps({"records": records}, indent=2)
        return "\n".join(f"- {r['input_id']} | {r['label']} | {r['next_action']} | line {r['evidence']}"
                         for r in records)

    def draft(self, inputs):
        """Pass-1 path (Part C). Freeform text with deliberate defects: a
        non-canonical label on I-2 and a too-short action on I-3."""
        lines = []
        for iid, text in inputs.items():
            label, ev = self.classify(text)
            action, _ = self.CANON[label]
            if iid == "I-2":
                label = "payment"
            if iid == "I-3":
                action = "Push the patch"
            lines.append(f"- {iid} | {label} | {action} | line {ev}")
        return "\n".join(lines)

    def generate(self, inputs, temperature=0.0, seed=0):
        """Structured path (Part E). temperature 0 is canonical and seed
        independent; temperature > 0 samples phrasing via a seeded RNG."""
        rng = random.Random(seed)
        records = []
        for iid, text in inputs.items():
            label, ev = self.classify(text)
            if temperature <= 0.0 or label == "unknown":
                action = self.CANON[label][0]
            else:
                action = rng.choice(self.PHRASE_POOL[label])
            records.append({"input_id": iid, "label": label,
                            "next_action": action, "evidence": str(ev)})
        return {"records": records}

model = MockTriageModel()
print("mock ready:", model.classify(INPUTS["I-1"]))


## Part A - Patterns and building blocks

The mock model classifies each support input into `account`, `billing`, `bug`,
or `unknown` using the K-Doc as its evidence source. You are not building the
classifier. You are building the reliability layer around it: the prompt
contract, the validator, the verify pass, the scorer, and the determinism probe.

**Delimiter palette.** Any consistent, machine-detectable fence works: triple
backticks, XML-style tags, YAML fences, or JSONL. This lab uses XML-style tags
because they nest cleanly and are trivial to detect with a regex:

```text
<INSTRUCTION> ... </INSTRUCTION>
<CONTEXT> ... </CONTEXT>
<INPUTS> ... </INPUTS>
<CONSTRAINTS> ... </CONSTRAINTS>
<CONTRACT> ... </CONTRACT>
```


## Part B - Delimiters and refusal guidance

### Task 1 - build a delimited prompt
Fill in `build_prompt` so it assembles one string with five clearly fenced
sections. Read the contract in the docstring carefully. The `<INPUTS>` block
must be shaped so the model can recover each id and its text.


In [ ]:
def build_prompt(instruction, context_lines, inputs, constraints, contract):
    """Assemble a single delimited prompt string (see contract in the stub)."""
    parts = []
    parts.append(f"<INSTRUCTION>\n{instruction}\n</INSTRUCTION>")
    ctx = "\n".join(f"{i + 1}. {line}" for i, line in enumerate(context_lines))
    parts.append(f"<CONTEXT>\n{ctx}\n</CONTEXT>")
    inp = "\n".join(f"{iid}: {text}" for iid, text in inputs.items())
    parts.append(f"<INPUTS>\n{inp}\n</INPUTS>")
    con = "\n".join(f"- {c}" for c in constraints)
    parts.append(f"<CONSTRAINTS>\n{con}\n</CONSTRAINTS>")
    parts.append(f"<CONTRACT>\n{json.dumps(contract, indent=2)}\n</CONTRACT>")
    return "\n\n".join(parts)


In [ ]:
# Build the prompt from the fixed lab data, then check its structure.
CONTRACT = {"records": [{"input_id": "<I-#>",
                         "label": "<account|billing|bug|unknown>",
                         "next_action": "<string 6 to 18 words>",
                         "evidence": "<K-Doc line number or none>"}]}
CONSTRAINTS = [
    "Exactly one next action per input, 6 to 18 words.",
    "Cite the K-Doc line number when evidence is used, otherwise none.",
    "If the label cannot be inferred, use unknown and ask for one specific detail.",
]
INSTRUCTION = "Classify each input into {account, billing, bug, unknown} and propose one next action."

TAGS = ["INSTRUCTION", "CONTEXT", "INPUTS", "CONSTRAINTS", "CONTRACT"]
PROMPT = None            # set by a working build_prompt; stays None until then

def _t1():
    p = build_prompt(INSTRUCTION, K_DOC, INPUTS, CONSTRAINTS, CONTRACT)
    global PROMPT
    PROMPT = p
    tags_ok = all(f"<{t}>" in p and f"</{t}>" in p for t in TAGS)
    inputs_ok = all(iid in p for iid in INPUTS)
    recovered = MockTriageModel()._extract_inputs(p)
    return tags_ok and inputs_ok and recovered == INPUTS

check("Task 1: all five delimiter blocks present", _t1)
check("Task 1: inputs recoverable from the INPUTS block",
      lambda: PROMPT is not None and MockTriageModel()._extract_inputs(PROMPT) == INPUTS)


### Task 2 - enforce the output contract
A delimiter keeps sections apart; a validator keeps the reply honest. Fill in
`validate_triage` to return a list of every violation it finds. An empty list
means the reply is clean.


In [ ]:
def validate_triage(data):
    """Return a list of error strings, empty when valid (see contract)."""
    if not isinstance(data, dict) or not isinstance(data.get("records"), list):
        return ["top-level object must contain a 'records' list"]
    if not data["records"]:
        return ["'records' list is empty"]
    errors = []
    for i, rec in enumerate(data["records"]):
        label = rec.get("label")
        if label not in ALLOWED:
            errors.append(f"[{i}] bad label: {label!r}")
        wc = len(WORD.findall(rec.get("next_action", "")))
        if wc < 6 or wc > 18:
            errors.append(f"[{i}] next_action word count {wc} out of range (6 to 18)")
        ev = str(rec.get("evidence", ""))
        if not (ev == "none" or re.fullmatch(r"[1-9]\d*", ev)):
            errors.append(f"[{i}] evidence must be a K-Doc line number or 'none', got {ev!r}")
    return errors


In [ ]:
_good = {"records": [{"input_id": "I-1", "label": "account",
    "next_action": "Ask the user to request a fresh reset link and retry within 24 hours",
    "evidence": "2"}]}
_bad = {"records": [{"input_id": "I-1", "label": "payment",
    "next_action": "Refund it", "evidence": "x"}]}

check("Task 2: clean record validates", lambda: validate_triage(_good) == [])
check("Task 2: bad record yields three errors", lambda: len(validate_triage(_bad)) == 3)
check("Task 2: missing records key reported",
      lambda: validate_triage({"foo": 1})[0].startswith("top-level"))
check("Task 2: empty records reported",
      lambda: validate_triage({"records": []}) == ["'records' list is empty"])


### Part B end to end
With both pieces in place, your prompt drives the mock model and your validator
confirms the reply. This is the smallest complete reliable prompt.


In [ ]:
# End-to-end for Part B: feed your prompt to the mock model, validate the reply.
_parsed = None
try:
    _reply = model.respond(PROMPT)
    _parsed = json.loads(_reply)
    print(json.dumps(_parsed, indent=2)[:400], "...")
except Exception as e:
    print("Part B not ready yet:", type(e).__name__)
check("Part B: model reply on your prompt is contract-valid",
      lambda: _parsed is not None and validate_triage(_parsed) == [])


## Part C - Answer to Verify

A single pass can be wrong in ways a delimiter cannot catch: a label outside the
allowed set, an action that is too terse to act on. The fix is a second pass
whose only job is to check the first against the contract and repair it.

### Task 3 - parse, verify, and repair
`model.draft(inputs)` gives you a deliberately flawed freeform draft. Fill in
`answer_then_verify` to parse it, repair every defect using the provided policy
tables, and return a contract-valid result.


In [ ]:
def _parse_draft(text):
    records = []
    for line in text.splitlines():
        line = line.strip().lstrip("-").strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) < 4:
            continue
        m = re.search(r"\d+", parts[3])
        records.append({"input_id": parts[0], "label": parts[1],
                        "next_action": parts[2],
                        "evidence": m.group(0) if m else "none"})
    return records

def answer_then_verify(model, inputs):
    """Two-pass Answer to Verify workflow (see contract in the stub)."""
    records = _parse_draft(model.draft(inputs))
    repaired = []
    for rec in records:
        label = LABEL_SYNONYMS.get(rec["label"].lower(), "unknown")
        action = rec["next_action"]
        if not (6 <= len(WORD.findall(action)) <= 18):
            action = FALLBACK_ACTIONS[label]
        ev = str(rec["evidence"])
        if not (ev == "none" or re.fullmatch(r"[1-9]\d*", ev)):
            ev = "none"
        repaired.append({"input_id": rec["input_id"], "label": label,
                         "next_action": action, "evidence": ev})
    result = {"records": repaired}
    assert validate_triage(result) == [], validate_triage(result)
    return result


In [ ]:
print("PASS 1 draft (deliberately flawed):")
print(model.draft(INPUTS))
FINAL = None
try:
    FINAL = answer_then_verify(model, INPUTS)
    print("\nPASS 2 repaired:")
    print(json.dumps(FINAL, indent=2))
except Exception as e:
    print("\nTask 3 not ready yet:", type(e).__name__)

_byid = {r["input_id"]: r for r in FINAL["records"]} if FINAL else {}
check("Task 3: final output is contract-valid",
      lambda: FINAL is not None and validate_triage(FINAL) == [])
check("Task 3: synonym label 'payment' repaired to 'billing'",
      lambda: _byid.get("I-2", {}).get("label") == "billing")
check("Task 3: too-short action repaired into range",
      lambda: 6 <= len(WORD.findall(_byid.get("I-3", {}).get("next_action", ""))) <= 18)


## Part D - Rubric-based scoring

Two ways to grade an output. First, a prompt that asks a model to score against
a rubric and return strict JSON (shown below as a reference artifact). Second, a
fast local scorer you can run in a pipeline with no model call. You will build
the second.

**Reference: a rubric scoring prompt.** In production you would send something
like this to a grader model, pasting the output under `<OUTPUT>` and the rubric
under `<RUBRIC>`, and require strict JSON back:

```text
<INSTRUCTION>
Score the OUTPUT against every criterion in RUBRIC. Return strict JSON only,
no prose, shaped as: {"score_total": <0-10>,
"scores": [{"criterion": "...", "score": <0-2>, "comment": "..."}],
"summary": "<one paragraph>"}.
</INSTRUCTION>
<RUBRIC> ... five criteria, 0 to 2 each ... </RUBRIC>
<OUTPUT> ... the triage JSON ... </OUTPUT>
```

### Task 4 - a lightweight auto rubric
Fill in `auto_rubric_score`. It grades the five criteria mechanically so you get
a fast numeric signal without a second model call.


In [ ]:
CLARIFY_WORDS = ("share", "provide", "specify", "confirm", "send", "exact", "digits", "details")

def _tiered(bad):
    return 2 if bad == 0 else (1 if bad == 1 else 0)

def auto_rubric_score(d):
    """Score a triage output 0 to 10 across five criteria (see contract)."""
    recs = d.get("records") if isinstance(d, dict) else None
    if not isinstance(recs, list) or not recs:
        fmt = 0
    else:
        fmt = 2 if {"input_id", "label", "next_action", "evidence"}.issubset(recs[0].keys()) else 1
    recs = recs if isinstance(recs, list) else []

    labels = _tiered(len([r for r in recs if r.get("label") not in ALLOWED]))
    rgx = re.compile(r"^(?:[1-9]\d*|none)$")
    evid = _tiered(len([r for r in recs if not rgx.fullmatch(str(r.get("evidence", "")))]))
    acts = _tiered(len([r for r in recs
                        if not (6 <= len(WORD.findall(r.get("next_action", ""))) <= 18)]))
    unknowns = [r for r in recs if r.get("label") == "unknown"]
    if not unknowns:
        refusal = 1
    else:
        good = [r for r in unknowns
                if any(w in r.get("next_action", "").lower() for w in CLARIFY_WORDS)]
        refusal = 2 if len(good) == len(unknowns) else (1 if good else 0)

    scores = [
        {"criterion": "Format", "score": fmt},
        {"criterion": "Label Validity", "score": labels},
        {"criterion": "Evidence Use", "score": evid},
        {"criterion": "Action Quality", "score": acts},
        {"criterion": "Refusal Discipline", "score": refusal},
    ]
    return {"score_total": sum(s["score"] for s in scores), "scores": scores}


In [ ]:
_weak = {"records": [
    {"input_id": "I-1", "label": "payment", "next_action": "Refund it", "evidence": "x"},
    {"input_id": "I-2", "label": "unknown", "next_action": "figure it out yourself", "evidence": "none"},
]}
try:
    _rg = auto_rubric_score(FINAL)
    print("repaired output scores", _rg["score_total"], "of 10")
    for s in _rg["scores"]:
        print("  ", s["criterion"], s["score"])
except Exception as e:
    _rg = None
    print("Task 4 not ready yet:", type(e).__name__)

check("Task 4: repaired output scores 8 or better",
      lambda: FINAL is not None and auto_rubric_score(FINAL)["score_total"] >= 8)
check("Task 4: weak output scores below the repaired one",
      lambda: _rg is not None and auto_rubric_score(_weak)["score_total"] < _rg["score_total"])
check("Task 4: returns five criteria",
      lambda: FINAL is not None and len(auto_rubric_score(FINAL)["scores"]) == 5)


## Part E - Determinism versus creativity

For pipelines and evaluation you want the same input to produce the same output.
For ideation you want variety. The lever is temperature (and its cousin top_p).

The mock makes this observable: at temperature 0 it returns one canonical
phrasing every time; above 0 it samples from a phrasing pool seeded by the run
index, so outputs vary.

A caution worth teaching. On real hosted endpoints temperature 0 is only
best-effort deterministic. A fixed `seed` is best effort too, is provider
specific, and Anthropic does not expose one. Treat determinism as a strong
default to request, not a guarantee to rely on.

### Task 5 - measure determinism
Fill in `compare_determinism` to count distinct outputs at two temperatures.


In [ ]:
def compare_determinism(model, inputs, runs=5):
    """Contrast reproducibility at two temperatures (see contract)."""
    def canon(result):
        return json.dumps(result, sort_keys=True)
    t0 = {canon(model.generate(inputs, temperature=0.0, seed=r)) for r in range(runs)}
    t7 = {canon(model.generate(inputs, temperature=0.7, seed=r)) for r in range(runs)}
    return {"runs": runs, "t0_unique": len(t0), "t7_unique": len(t7)}


In [ ]:
_cmp = None
try:
    _cmp = compare_determinism(model, INPUTS, runs=5)
    print(_cmp)
except Exception as e:
    print("Task 5 not ready yet:", type(e).__name__)
check("Task 5: temperature 0 is reproducible (one unique output)",
      lambda: _cmp is not None and _cmp["t0_unique"] == 1)
check("Task 5: temperature 0.7 varies (more than one unique output)",
      lambda: _cmp is not None and _cmp["t7_unique"] > 1)


## Part F - Wrap-up and checks

### Knowledge checks
1. Why do delimiters reduce leakage and improve parseability?
2. Give a case where refusing is safer than guessing.
3. What is the purpose of the verify pass in Answer to Verify?
4. When would you lower temperature versus reduce top_p?

Run `summary()` below to confirm all core checks pass before you move on.


In [ ]:
summary()

## Stretch goals

Optional, for fast finishers. Each has its own checks. Reference solutions are
in the instructor solution notebook.

### Stretch 1 - ambiguity-aware refusal
Real tickets carry mixed signals. Make classification refuse to unknown when the
text matches two or more categories, rather than letting keyword order decide.

### Stretch 2 - a strict action template
Force every next_action into an `Action: ... Because: ...` shape and extend the
validator to enforce it.

### Stretch 3 - batch drift report
Quantify which inputs are unstable under creativity by counting distinct
phrasings per input across several temperature 0.7 runs.


In [ ]:
CATEGORY_KEYWORDS = {
    "billing": ("charge", "charged", "refund", "billing", "card"),
    "bug": ("crash", "export", "android", "pixel"),
    "account": ("reset", "token", "password", "login", "link"),
}

def classify_with_ambiguity(text):
    """Return (label, evidence); mixed signals refuse to unknown (see contract)."""
    low = text.lower()
    hits = [cat for cat, kws in CATEGORY_KEYWORDS.items() if any(k in low for k in kws)]
    if len(hits) == 1:
        return hits[0], {"billing": 3, "bug": 4, "account": 2}[hits[0]]
    return "unknown", "none"


In [ ]:
check("Stretch 1: single-signal input still classifies",
      lambda: classify_with_ambiguity("Reset link token invalid")[0] == "account")
check("Stretch 1: mixed signals route to unknown",
      lambda: classify_with_ambiguity(
          "I was double charged and now the app crashes on export") == ("unknown", "none"))


In [ ]:
TEMPLATE = re.compile(r"^Action:\s+.+?\s+Because:\s+.+$")

def templated_action(label):
    """Return a template-conforming action for label (see contract)."""
    return {
        "account": "Action: send a fresh reset link Because: the prior link expired after 24 hours",
        "billing": "Action: escalate the duplicate charge Because: it stayed unresolved past 48 hours",
        "bug": "Action: install the v5.2.1 patch Because: it fixes the Android export crash",
        "unknown": "Action: request the exact error text Because: the current signal is insufficient",
    }[label]

def validate_templated(data):
    """validate_triage plus a template check on every action (see contract)."""
    errors = validate_triage(data)
    if errors and errors[0].startswith("top-level"):
        return errors
    for i, rec in enumerate(data.get("records", [])):
        if not TEMPLATE.match(rec.get("next_action", "")):
            errors.append(f"[{i}] next_action must match 'Action: ... Because: ...'")
    return errors


In [ ]:
_tbad = {"records": [{"input_id": "I-1", "label": "account",
                      "next_action": "send a new link", "evidence": "2"}]}

def _build_tgood():
    recs = [{"input_id": iid, "label": lbl, "next_action": templated_action(lbl), "evidence": ev}
            for iid, lbl, ev in (("I-1", "account", "2"), ("I-2", "billing", "3"), ("I-3", "bug", "4"))]
    return {"records": recs}

check("Stretch 2: templated actions pass the templated validator",
      lambda: validate_templated(_build_tgood()) == [])
check("Stretch 2: non-template action is caught",
      lambda: any("Action:" in e for e in validate_templated(_tbad)))
check("Stretch 2: templated actions stay in the 6 to 18 word window",
      lambda: all(6 <= len(WORD.findall(templated_action(l))) <= 18
                  for l in ("account", "billing", "bug", "unknown")))


In [ ]:
def drift_report(model, inputs, runs=8):
    """Distinct phrasings per input across creative runs (see contract)."""
    seen = {iid: set() for iid in inputs}
    for r in range(runs):
        for rec in model.generate(inputs, temperature=0.7, seed=r)["records"]:
            seen[rec["input_id"]].add(rec["next_action"])
    return {iid: len(v) for iid, v in seen.items()}


In [ ]:
try:
    _drift = drift_report(model, INPUTS, runs=8)
    print("distinct phrasings per input at temperature 0.7:", _drift)
except Exception as e:
    print("Stretch 3 not ready yet:", type(e).__name__)
check("Stretch 3: every input drifts under creativity",
      lambda: all(v > 1 for v in drift_report(model, INPUTS, runs=8).values()))


## From mock to production

The reliability layer you built does not change when the model becomes real.
`build_prompt` still assembles the request, `validate_triage` still guards the
reply, and Answer to Verify still repairs it. The cell below shows the wiring to
the Anthropic Messages API using generally available structured outputs. It is
reference only and is not executed here.


In [ ]:
# --- Reference only: wiring the same pattern to the real Anthropic API ---
# This cell defines the function but does NOT call it, so the notebook stays
# offline. Set RUN_LIVE = True only in your own environment with a real key.
#
# CURRENCY FLAG: structured outputs are generally available on the Claude API
# via output_config.format (no beta header). Confirm the current model id and
# that output_config.format is still the GA shape before class:
# https://platform.claude.com/docs/en/build-with-claude/structured-outputs
import os

RUN_LIVE = False
MODEL = os.environ.get("TRIAGE_MODEL", "claude-sonnet-4-6")  # confirm current id

TRIAGE_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {"records": {"type": "array", "items": {
        "type": "object", "additionalProperties": False,
        "properties": {
            "input_id": {"type": "string"},
            "label": {"type": "string", "enum": ["account", "billing", "bug", "unknown"]},
            "next_action": {"type": "string"},
            "evidence": {"type": "string"},
        },
        "required": ["input_id", "label", "next_action", "evidence"],
    }}},
    "required": ["records"],
}

def triage_live(prompt, model_id=MODEL):
    """Send a built prompt to the Anthropic Messages API and return parsed JSON.
    The prompt is the exact string build_prompt produced. Structure is enforced
    server-side by the JSON schema, then validate_triage is your local guard."""
    from anthropic import Anthropic          # pip install anthropic
    client = Anthropic()                      # reads ANTHROPIC_API_KEY from env
    msg = client.messages.create(
        model=model_id,
        max_tokens=1024,
        temperature=0,                        # deterministic path for a pipeline
        messages=[{"role": "user", "content": prompt}],
        output_config={"format": {"type": "json_schema", "schema": TRIAGE_SCHEMA}},
    )
    text = next(b.text for b in msg.content if b.type == "text")
    data = json.loads(text)
    problems = validate_triage(data)          # never trust structure alone
    if problems:
        raise ValueError(f"schema passed but policy failed: {problems}")
    return data

if RUN_LIVE:
    print(json.dumps(triage_live(PROMPT), indent=2))
else:
    print("Live call skipped (RUN_LIVE is False). See the instructor walkthrough to run it.")
